In [1]:
!pip install pandas numpy matplotlib scikit-learn lifelines scikit-survival openpyxl

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 350.0/350.0 kB 6.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 54.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 103.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 14.0 MB/s eta 0:00:00
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4030 sha256=28a0fc7e719381937aa04c151a75f5684049f09947250d3130d18eb24accaa87
  Stored in directory: /root/.cache/pip/wheels/50/37/21/0a719b9d89c635e89ff24bd93b862882ad675279552013b2fb
Successfully built autograd-gamma
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [2]:
# ==========================================
# CELL 1: Install & Import Libraries (GBSA)
# ==========================================

# Uncomment if first time running
# !pip install pandas numpy matplotlib scikit-learn lifelines scikit-survival openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer

# GBSA model
from sksurv.ensemble import GradientBoostingSurvivalAnalysis

# Survival utilities
from sksurv.util import Surv

# Evaluation metrics
from sksurv.metrics import (
    concordance_index_censored,
    cumulative_dynamic_auc,
    integrated_brier_score
)

import warnings
warnings.filterwarnings("ignore")

# Global configuration
N_REPEATS = 25
RANDOM_STATE_BASE = 42
TIME_GRID = np.arange(0.25, 5.0, 0.25)

print("GBSA Survival Pipeline Initialized")

GBSA Survival Pipeline Initialized


In [5]:
# ==========================================
# CELL 2: Load & Preprocess Data (GBSA)
# ==========================================

file_path = r"/kaggle/input/datasets/swetajnathan30/umhs-dataset/Training.xlsx"

data = pd.read_excel(file_path)

# ------------------------------------------
# Encode categorical variables
# ------------------------------------------
data["Race"] = np.where(data["Race"] == "Caucasian", 1, 0)
data["Smoking"] = np.where(data["Smoking"] == "Current", 1, 0)
data["Alcohol"] = np.where(data["Alcohol"] == "Yes", 1, 0)
data["Drug"] = np.where(data["Drug"] == "Yes", 1, 0)

# ------------------------------------------
# Convert numeric columns safely
# ------------------------------------------
numeric_cols = ["HGB", "Duration", "EndEvent", "MAGGIC"]

for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors="coerce")

# ------------------------------------------
# Create survival columns
# ------------------------------------------
data["time"] = data["Duration"]
data["event"] = data["EndEvent"]

# Remove original columns
data.drop(columns=["Duration", "EndEvent"], inplace=True)

# Remove duplicate columns if any
data = data.loc[:, ~data.columns.duplicated()]

print("Data Loaded Successfully")
print("Shape:", data.shape)


Data Loaded Successfully
Shape: (343, 157)


In [8]:
# ==========================================
# CELL 3: Create D, M, DM subsets (GBSA)
# ==========================================

# Get column list
cols = list(data.columns)

# Find index positions
idx_race = cols.index("Race")
idx_c_sa = cols.index("C_SA")
idx_rvpower_s = cols.index("RVpowerIndex_S")

# ------------------------------------------
# Digital features subset (D)
# ------------------------------------------
D = pd.concat([
    data.iloc[:, idx_race:idx_c_sa],
    data[["time", "event"]]
], axis=1)

# ------------------------------------------
# MRI features subset (M)
# ------------------------------------------
M = pd.concat([
    data.iloc[:, idx_c_sa:idx_rvpower_s + 1],
    data[["time", "event"]]
], axis=1)

# ------------------------------------------
# Combined features subset (DM)
# ------------------------------------------
DM = pd.concat([
    data.iloc[:, idx_race:idx_rvpower_s + 1],
    data[["time", "event"]]
], axis=1)

# ------------------------------------------
# Remove duplicate columns (important)
# ------------------------------------------
D = D.loc[:, ~D.columns.duplicated()]
M = M.loc[:, ~M.columns.duplicated()]
DM = DM.loc[:, ~DM.columns.duplicated()]

# ------------------------------------------
# Print shapes
# ------------------------------------------
print("D:", D.shape)
print("M:", M.shape)
print("DM:", DM.shape)

D: (343, 69)
M: (343, 82)
DM: (343, 149)


In [11]:
from collections import defaultdict
from sklearn.inspection import permutation_importance
from sklearn.impute import KNNImputer
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.util import Surv

def gbsa_variable_hunting(dataset, model_name, n_repeats=25):

    print(f"\nStarting Variable Hunting for {model_name} (GBSA)")

    freq_counter = defaultdict(int)
    importance_counter = defaultdict(float)

    X_full = dataset.drop(columns=["time", "event"])
    y_full = Surv.from_dataframe("event", "time", dataset)

    # FIX: Impute missing values
    imputer = KNNImputer(n_neighbors=5)
    X_full_imputed = pd.DataFrame(
        imputer.fit_transform(X_full),
        columns=X_full.columns
    )

    for rep in range(n_repeats):

        print(f"{model_name} Repeat {rep+1}/{n_repeats}")

        gbsa = GradientBoostingSurvivalAnalysis(
            n_estimators=100,
            learning_rate=0.05,
            max_depth=3,
            random_state=RANDOM_STATE_BASE + rep
        )

        gbsa.fit(X_full_imputed, y_full)

        perm = permutation_importance(
            gbsa,
            X_full_imputed,
            y_full,
            n_repeats=5,
            random_state=RANDOM_STATE_BASE + rep,
            n_jobs=-1
        )

        importances = perm.importances_mean

        importance_dict = dict(zip(X_full_imputed.columns, importances))

        threshold = np.mean(importances)

        selected_features = [
            feature for feature, imp in importance_dict.items()
            if imp >= threshold
        ]

        for feature in selected_features:
            freq_counter[feature] += 1
            importance_counter[feature] += importance_dict[feature]

    avg_importance = {
        feature: importance_counter[feature] / n_repeats
        for feature in importance_counter
    }

    sorted_features = sorted(
        avg_importance.items(),
        key=lambda x: x[1],
        reverse=True
    )

    sorted_features_dict = dict(sorted_features)

    print(f"\nCompleted Variable Hunting for {model_name} (GBSA)")

    return {
        "frequency": dict(freq_counter),
        "importance": sorted_features_dict
    }

In [12]:
result_D = gbsa_variable_hunting(D, "D", N_REPEATS)
result_M = gbsa_variable_hunting(M, "M", N_REPEATS)
result_DM = gbsa_variable_hunting(DM, "DM", N_REPEATS)


Starting Variable Hunting for D (GBSA)
D Repeat 1/25
D Repeat 2/25
D Repeat 3/25
D Repeat 4/25
D Repeat 5/25
D Repeat 6/25
D Repeat 7/25
D Repeat 8/25
D Repeat 9/25
D Repeat 10/25
D Repeat 11/25
D Repeat 12/25
D Repeat 13/25
D Repeat 14/25
D Repeat 15/25
D Repeat 16/25
D Repeat 17/25
D Repeat 18/25
D Repeat 19/25
D Repeat 20/25
D Repeat 21/25
D Repeat 22/25
D Repeat 23/25
D Repeat 24/25
D Repeat 25/25

Completed Variable Hunting for D (GBSA)

Starting Variable Hunting for M (GBSA)
M Repeat 1/25
M Repeat 2/25
M Repeat 3/25
M Repeat 4/25
M Repeat 5/25
M Repeat 6/25
M Repeat 7/25
M Repeat 8/25
M Repeat 9/25
M Repeat 10/25
M Repeat 11/25
M Repeat 12/25
M Repeat 13/25
M Repeat 14/25
M Repeat 15/25
M Repeat 16/25
M Repeat 17/25
M Repeat 18/25
M Repeat 19/25
M Repeat 20/25
M Repeat 21/25
M Repeat 22/25
M Repeat 23/25
M Repeat 24/25
M Repeat 25/25

Completed Variable Hunting for M (GBSA)

Starting Variable Hunting for DM (GBSA)
DM Repeat 1/25
DM Repeat 2/25
DM Repeat 3/25
DM Repeat 4/25
DM Re

In [13]:
# ==========================================
# CELL: Print Top Features (GBSA)
# ==========================================

print("\nTop 10 features in D:")
for feature, importance in list(result_D["importance"].items())[:10]:
    print(f"{feature}: {importance:.6f}")


print("\nTop 10 features in M:")
for feature, importance in list(result_M["importance"].items())[:10]:
    print(f"{feature}: {importance:.6f}")


print("\nTop 10 features in DM:")
for feature, importance in list(result_DM["importance"].items())[:10]:
    print(f"{feature}: {importance:.6f}")


Top 10 features in D:
HGB: 0.103273
Creatinine: 0.042450
EAr_D: 0.039959
CO_D: 0.032183
Age: 0.012952
Septal E/e': 0.008459
LVEDV_D: 0.007239
RVm_D: 0.006365
LVESP_D: 0.003221
RVpower_D: 0.003066

Top 10 features in M:
EAr_S: 0.029108
PVR_S: 0.021248
C_SV: 0.019065
B_P: 0.018893
TVr_S: 0.015377
PADP_S: 0.012887
MVmg_S: 0.012152
LVpower_S: 0.009582
K_P: 0.009188
LVESV_S: 0.006503

Top 10 features in DM:
HGB: 0.087479
Creatinine: 0.027481
PVR_S: 0.018544
PVR_D: 0.009084
EAr_S: 0.007892
LVpower_S: 0.006221
Age: 0.005922
MVmg_S: 0.004864
Vh0: 0.004547
Vw_LV: 0.004407


In [14]:
result_D
result_M
result_DM

{'frequency': {'Age': 25,
  'HGB': 25,
  'Creatinine': 25,
  'DBP_D': 18,
  'CO_D': 23,
  'LVEDV_D': 21,
  'PVR_D': 25,
  'RVpower_D': 21,
  'Vw_LV': 25,
  'Amref_SEP': 11,
  'Vh0': 25,
  'K_P': 25,
  'B_P': 9,
  'R_tSA': 25,
  'R_RA': 17,
  'R_p_o': 25,
  'EF_S': 9,
  'TVr_S': 9,
  'MVmg_S': 25,
  'PVpg_S': 24,
  'EAr_S': 25,
  'LVpower_S': 25,
  'PVR_S': 25,
  'C_SV': 10,
  'SVR_S': 24,
  'R_LA': 6,
  'StrainLV_S': 10,
  'LAVmax_D': 2,
  'SBP_D': 2,
  'StressRV_S': 2,
  'C_PV': 3},
 'importance': {'HGB': np.float64(0.08747883982009413),
  'Creatinine': np.float64(0.02748082303360833),
  'PVR_S': np.float64(0.018543896306264775),
  'PVR_D': np.float64(0.009084251159825708),
  'EAr_S': np.float64(0.007892481495909563),
  'LVpower_S': np.float64(0.006221340793993644),
  'Age': np.float64(0.005922300527676397),
  'MVmg_S': np.float64(0.004864397775967652),
  'Vh0': np.float64(0.004547225271806446),
  'Vw_LV': np.float64(0.004406700428515726),
  'R_p_o': np.float64(0.0038012536742571246),

In [15]:
# ==========================================
# CELL: Extract Feature Groups (GBSA)
# ==========================================

def extract_feature_groups(result, n_repeats=25):

    freq = result["frequency"]
    importance = result["importance"]

    # Mandatory features: ≥80% frequency
    mandatory = [
        feature for feature, count in freq.items()
        if count >= 0.8 * n_repeats
    ]

    # Valid features: ≥5 repeats
    valid = [
        feature for feature, count in freq.items()
        if count >= 5
    ]

    # Sort valid features by importance
    sorted_valid = {
        feature: importance[feature]
        for feature in valid
        if feature in importance
    }

    sorted_valid = dict(
        sorted(sorted_valid.items(),
               key=lambda x: x[1],
               reverse=True)
    )

    return mandatory, valid, sorted_valid


# Apply to GBSA results
mandatory_D, valid_D, sorted_valid_D = extract_feature_groups(result_D, N_REPEATS)
mandatory_M, valid_M, sorted_valid_M = extract_feature_groups(result_M, N_REPEATS)
mandatory_DM, valid_DM, sorted_valid_DM = extract_feature_groups(result_DM, N_REPEATS)


print("\nMandatory D:", len(mandatory_D))
print("Valid D:", len(valid_D))

print("\nMandatory M:", len(mandatory_M))
print("Valid M:", len(valid_M))

print("\nMandatory DM:", len(mandatory_DM))
print("Valid DM:", len(valid_DM))


Mandatory D: 8
Valid D: 10

Mandatory M: 15
Valid M: 21

Mandatory DM: 18
Valid DM: 27


In [17]:
# ==========================================
# CELL: Train GBSA model and compute C-index
# ==========================================

from sklearn.impute import KNNImputer
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored

def train_gbsa_model(dataset, features, random_state):

    X = dataset[features]
    y = Surv.from_dataframe("event", "time", dataset)

    # FIX: Impute missing values (required for GBSA)
    imputer = KNNImputer(n_neighbors=5)

    X_imputed = pd.DataFrame(
        imputer.fit_transform(X),
        columns=X.columns
    )

    gbsa = GradientBoostingSurvivalAnalysis(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        random_state=random_state
    )

    gbsa.fit(X_imputed, y)

    risk_scores = gbsa.predict(X_imputed)

    cindex = concordance_index_censored(
        dataset["event"].astype(bool),
        dataset["time"],
        risk_scores
    )[0]

    return cindex

In [18]:
# ==========================================
# CELL: Forward Feature Selection (GBSA)
# ==========================================

def find_best_features_gbsa(dataset, sorted_valid_features, mandatory_features,
                            model_name, max_additional=12, repeats=10):

    print(f"\nSelecting best features for {model_name} (GBSA)")

    selected = mandatory_features.copy()
    remaining = list(sorted_valid_features.keys())

    remaining = [f for f in remaining if f not in selected]

    history = []

    # Evaluate mandatory features first
    scores = []

    for rep in range(repeats):

        cindex = train_gbsa_model(
            dataset,
            selected,
            RANDOM_STATE_BASE + rep
        )

        scores.append(cindex)

    best_score = np.mean(scores)

    history.append((len(selected), best_score, selected.copy()))

    print(f"Initial features: {len(selected)} | C-index: {best_score:.4f}")

    # Forward selection
    for i in range(max_additional):

        best_feature = None
        best_feature_score = best_score

        for feature in remaining:

            test_features = selected + [feature]

            scores = []

            for rep in range(repeats):

                cindex = train_gbsa_model(
                    dataset,
                    test_features,
                    RANDOM_STATE_BASE + rep
                )

                scores.append(cindex)

            mean_score = np.mean(scores)

            if mean_score > best_feature_score:

                best_feature_score = mean_score
                best_feature = feature

        if best_feature is None:
            break

        selected.append(best_feature)
        remaining.remove(best_feature)
        best_score = best_feature_score

        history.append((len(selected), best_score, selected.copy()))

        print(f"Added: {best_feature} | Total: {len(selected)} | C-index: {best_score:.4f}")

    return selected, history

In [19]:
selected_D, history_D = find_best_features_gbsa(
    D, sorted_valid_D, mandatory_D, "D"
)

selected_M, history_M = find_best_features_gbsa(
    M, sorted_valid_M, mandatory_M, "M"
)

selected_DM, history_DM = find_best_features_gbsa(
    DM, sorted_valid_DM, mandatory_DM, "DM"
)

print("\nFinal selected features:")
print("D:", len(selected_D))
print("M:", len(selected_M))
print("DM:", len(selected_DM))


Selecting best features for D (GBSA)
Initial features: 8 | C-index: 0.8644
Added: LVESP_D | Total: 9 | C-index: 0.8674
Added: RVpower_D | Total: 10 | C-index: 0.8675

Selecting best features for M (GBSA)
Initial features: 15 | C-index: 0.8913
Added: LAVmax_S | Total: 16 | C-index: 0.8958
Added: R_LA | Total: 17 | C-index: 0.8969

Selecting best features for DM (GBSA)
Initial features: 18 | C-index: 0.8957
Added: R_RA | Total: 19 | C-index: 0.9021
Added: DBP_D | Total: 20 | C-index: 0.9090
Added: B_P | Total: 21 | C-index: 0.9098

Final selected features:
D: 10
M: 17
DM: 21


In [31]:
# ==========================================
# STEP 1: Train-Test Split for D, M, DM
# ==========================================

from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sksurv.util import Surv

def split_dataset_gbsa(dataset, selected_features, model_name, test_size=0.3):

    print(f"\nSplitting dataset for {model_name}")

    # Select features
    X = dataset[selected_features]

    # Survival target
    y = Surv.from_dataframe("event", "time", dataset)

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=42
    )

    # Imputation (fit on TRAIN only)
    imputer = KNNImputer(n_neighbors=5)

    X_train = pd.DataFrame(
        imputer.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index
    )

    X_test = pd.DataFrame(
        imputer.transform(X_test),
        columns=X_test.columns,
        index=X_test.index
    )

    print(f"{model_name} Train shape:", X_train.shape)
    print(f"{model_name} Test shape:", X_test.shape)

    return X_train, X_test, y_train, y_test


# ==========================================
# Run for Digital (D)
# ==========================================

X_train_D, X_test_D, y_train_D, y_test_D = split_dataset_gbsa(
    D,
    selected_D,
    "D"
)


# ==========================================
# Run for MRI (M)
# ==========================================

X_train_M, X_test_M, y_train_M, y_test_M = split_dataset_gbsa(
    M,
    selected_M,
    "M"
)


# ==========================================
# Run for Combined (DM)
# ==========================================

X_train_DM, X_test_DM, y_train_DM, y_test_DM = split_dataset_gbsa(
    DM,
    selected_DM,
    "DM"
)


Splitting dataset for D
D Train shape: (240, 10)
D Test shape: (103, 10)

Splitting dataset for M
M Train shape: (240, 17)
M Test shape: (103, 17)

Splitting dataset for DM
DM Train shape: (240, 21)
DM Test shape: (103, 21)


In [33]:
# ==========================================
# CELL: Hyperparameter tuning with Cross-Validation (GBSA)
# ==========================================

from sklearn.model_selection import KFold, ParameterGrid
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.metrics import concordance_index_censored

def tune_gbsa_cv(X_train, y_train, model_name, n_splits=5):

    print(f"\nHyperparameter tuning with CV for {model_name}")

    param_grid = {
        "n_estimators": [100, 300, 500],
        "learning_rate": [0.01, 0.05, 0.1],
        "max_depth": [1, 2, 3],
        "subsample": [0.6, 0.8]
    }

    kf = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    best_score = -1
    best_params = None

    for params in ParameterGrid(param_grid):

        fold_scores = []

        for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):

            X_fold_train = X_train.iloc[train_idx]
            X_fold_val   = X_train.iloc[val_idx]

            y_fold_train = y_train[train_idx]
            y_fold_val   = y_train[val_idx]

            gbsa = GradientBoostingSurvivalAnalysis(
                **params,
                random_state=42
            )

            # Train on fold training data
            gbsa.fit(X_fold_train, y_fold_train)

            # Predict on fold validation data
            risk_scores = gbsa.predict(X_fold_val)

            cindex = concordance_index_censored(
                y_fold_val["event"],
                y_fold_val["time"],
                risk_scores
            )[0]

            fold_scores.append(cindex)

        mean_score = np.mean(fold_scores)
        std_score  = np.std(fold_scores)

        print(f"{params} → Mean CV C-index: {mean_score:.4f} (±{std_score:.4f})")

        if mean_score > best_score:

            best_score = mean_score
            best_params = params

    print(f"\nBest params for {model_name}:")
    print(best_params)
    print(f"Best CV C-index: {best_score:.4f}")

    return best_params

In [34]:
# Run CV tuning
best_params_D  = tune_gbsa_cv(X_train_D,  y_train_D,  "D")
best_params_M  = tune_gbsa_cv(X_train_M,  y_train_M,  "M")
best_params_DM = tune_gbsa_cv(X_train_DM, y_train_DM, "DM")


Hyperparameter tuning with CV for D
{'learning_rate': 0.01, 'max_depth': 1, 'n_estimators': 100, 'subsample': 0.6} → Mean CV C-index: 0.6782 (±0.1001)
{'learning_rate': 0.01, 'max_depth': 1, 'n_estimators': 100, 'subsample': 0.8} → Mean CV C-index: 0.6676 (±0.1057)
{'learning_rate': 0.01, 'max_depth': 1, 'n_estimators': 300, 'subsample': 0.6} → Mean CV C-index: 0.6911 (±0.0932)
{'learning_rate': 0.01, 'max_depth': 1, 'n_estimators': 300, 'subsample': 0.8} → Mean CV C-index: 0.6816 (±0.0929)
{'learning_rate': 0.01, 'max_depth': 1, 'n_estimators': 500, 'subsample': 0.6} → Mean CV C-index: 0.6994 (±0.0879)
{'learning_rate': 0.01, 'max_depth': 1, 'n_estimators': 500, 'subsample': 0.8} → Mean CV C-index: 0.6949 (±0.0945)
{'learning_rate': 0.01, 'max_depth': 2, 'n_estimators': 100, 'subsample': 0.6} → Mean CV C-index: 0.7054 (±0.0880)
{'learning_rate': 0.01, 'max_depth': 2, 'n_estimators': 100, 'subsample': 0.8} → Mean CV C-index: 0.6852 (±0.0900)
{'learning_rate': 0.01, 'max_depth': 2, 'n_

In [36]:
from sksurv.metrics import cumulative_dynamic_auc, integrated_brier_score

def train_and_evaluate_test_gbsa(
    X_train, y_train,
    X_test, y_test,
    best_params,
    model_name
):

    print(f"\nFinal GBSA training and testing for {model_name}")

    # Train model on TRAIN data
    gbsa = GradientBoostingSurvivalAnalysis(
        **best_params,
        random_state=42
    )

    gbsa.fit(X_train, y_train)

    # Predict on TEST data
    risk_scores = gbsa.predict(X_test)

    # ==========================================
    # C-index
    # ==========================================
    cindex = concordance_index_censored(
        y_test["event"],
        y_test["time"],
        risk_scores
    )[0]

    # ==========================================
    # Dynamic time grid
    # ==========================================
    max_time = min(y_train["time"].max(), y_test["time"].max())

    valid_times = np.linspace(
        y_test["time"].min() + 1e-6,
        max_time * 0.99,
        20
    )

    # ==========================================
    # iAUC
    # ==========================================
    auc_times, auc_scores = cumulative_dynamic_auc(
        y_train,
        y_test,
        risk_scores,
        valid_times
    )

    iauc = np.mean(auc_scores)

    # ==========================================
    # Brier Score
    # ==========================================
    surv_funcs = gbsa.predict_survival_function(X_test)

    surv_matrix = np.row_stack([
        fn(valid_times) for fn in surv_funcs
    ])

    brier = integrated_brier_score(
        y_train,
        y_test,
        surv_matrix,
        valid_times
    )

    # Print results
    print(f"\n{model_name} TEST Results:")
    print(f"C-index: {cindex:.4f}")
    print(f"iAUC:    {iauc:.4f}")
    print(f"Brier:   {brier:.4f}")

    return gbsa, cindex, iauc, brier

In [37]:
model_D, cindex_D, iauc_D, brier_D = train_and_evaluate_test_gbsa(
    X_train_D, y_train_D,
    X_test_D, y_test_D,
    best_params_D,
    "D"
)

model_M, cindex_M, iauc_M, brier_M = train_and_evaluate_test_gbsa(
    X_train_M, y_train_M,
    X_test_M, y_test_M,
    best_params_M,
    "M"
)

model_DM, cindex_DM, iauc_DM, brier_DM = train_and_evaluate_test_gbsa(
    X_train_DM, y_train_DM,
    X_test_DM, y_test_DM,
    best_params_DM,
    "DM"
)


Final GBSA training and testing for D

D TEST Results:
C-index: 0.7072
iAUC:    0.6996
Brier:   0.1510

Final GBSA training and testing for M

M TEST Results:
C-index: 0.6518
iAUC:    0.6990
Brier:   0.1855

Final GBSA training and testing for DM

DM TEST Results:
C-index: 0.6645
iAUC:    0.6758
Brier:   0.1674


In [38]:
test_data = pd.read_excel("/kaggle/input/datasets/swetajnathan30/umhs-dataset/Testing.xlsx")

# Apply SAME preprocessing
test_data["Race"] = np.where(test_data["Race"] == "Caucasian", 1, 0)
test_data["Smoking"] = np.where(test_data["Smoking"] == "Current", 1, 0)
test_data["Alcohol"] = np.where(test_data["Alcohol"] == "Yes", 1, 0)
test_data["Drug"] = np.where(test_data["Drug"] == "Yes", 1, 0)

test_data["time"] = test_data["Duration"]
test_data["event"] = test_data["EndEvent"]

test_data.drop(columns=["Duration", "EndEvent"], inplace=True)

In [41]:
# ==========================================
# CELL: Final GBSA training and evaluation on EXTERNAL dataset (FIXED)
# ==========================================

from sklearn.impute import KNNImputer
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.metrics import (
    concordance_index_censored,
    cumulative_dynamic_auc,
    integrated_brier_score
)

def train_and_evaluate_external_gbsa(train_dataset, test_dataset,
                                     selected_features, best_params,
                                     model_name):

    print(f"\nFINAL GBSA Training and External Evaluation: {model_name}")

    # ==========================================
    # Prepare training data
    # ==========================================
    X_train = train_dataset[selected_features].copy()
    y_train = Surv.from_dataframe("event", "time", train_dataset)

    # ==========================================
    # Prepare test data
    # ==========================================
    X_test = test_dataset[selected_features].copy()
    y_test = Surv.from_dataframe("event", "time", test_dataset)

    # ==========================================
    # Apply KNN Imputation (FIT on train, TRANSFORM on test)
    # ==========================================
    imputer = KNNImputer(n_neighbors=5)

    X_train = pd.DataFrame(
        imputer.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index
    )

    X_test = pd.DataFrame(
        imputer.transform(X_test),
        columns=X_test.columns,
        index=X_test.index
    )

    # ==========================================
    # Train GBSA model
    # ==========================================
    gbsa = GradientBoostingSurvivalAnalysis(
        **best_params,
        random_state=42
    )

    gbsa.fit(X_train, y_train)

    # ==========================================
    # Predict on external test set
    # ==========================================
    risk_scores = gbsa.predict(X_test)

    # C-index
    cindex = concordance_index_censored(
        y_test["event"],
        y_test["time"],
        risk_scores
    )[0]

    # Time grid
    max_time = min(y_train["time"].max(), y_test["time"].max())

    valid_times = np.linspace(
        y_test["time"].min() + 1e-6,
        max_time * 0.99,
        20
    )

    # iAUC
    _, auc_scores = cumulative_dynamic_auc(
        y_train,
        y_test,
        risk_scores,
        valid_times
    )

    iauc = np.mean(auc_scores)

    # Brier score
    surv_funcs = gbsa.predict_survival_function(X_test)

    surv_matrix = np.row_stack([
        fn(valid_times) for fn in surv_funcs
    ])

    brier = integrated_brier_score(
        y_train,
        y_test,
        surv_matrix,
        valid_times
    )

    print(f"\n{model_name} External Results (GBSA):")
    print(f"C-index: {cindex:.4f}")
    print(f"iAUC:    {iauc:.4f}")
    print(f"Brier:   {brier:.4f}")

    return gbsa, cindex, iauc, brier

In [42]:
model_D_gbsa, cindex_D_gbsa, iauc_D_gbsa, brier_D_gbsa = train_and_evaluate_external_gbsa(
    data, test_data,
    selected_D,
    best_params_D,
    "D"
)

model_M_gbsa, cindex_M_gbsa, iauc_M_gbsa, brier_M_gbsa = train_and_evaluate_external_gbsa(
    data, test_data,
    selected_M,
    best_params_M,
    "M"
)

model_DM_gbsa, cindex_DM_gbsa, iauc_DM_gbsa, brier_DM_gbsa = train_and_evaluate_external_gbsa(
    data, test_data,
    selected_DM,
    best_params_DM,
    "DM"
)


FINAL GBSA Training and External Evaluation: D

D External Results (GBSA):
C-index: 0.5198
iAUC:    0.5350
Brier:   0.2464

FINAL GBSA Training and External Evaluation: M

M External Results (GBSA):
C-index: 0.5727
iAUC:    0.5594
Brier:   0.2144

FINAL GBSA Training and External Evaluation: DM

DM External Results (GBSA):
C-index: 0.5906
iAUC:    0.5889
Brier:   0.2065
